# 03. Model Coxa i diagnostyka

Notebook obejmuje model Coxa, test PH, reszty Schoenfelda, obserwacje wpływowe oraz model z interakcją czasową dla frakcji wyrzutowej.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, NelsonAalenFitter, WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter, ExponentialFitter, CoxPHFitter
from lifelines.statistics import logrank_test, proportional_hazard_test

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "Data" / "heart_failure_clinical_records_dataset.csv"
WORKSPACE = BASE_DIR
TABLES = WORKSPACE / "Tables"
FIGS = WORKSPACE / "Wykresy"

df = pd.read_csv(DATA_PATH).drop_duplicates()
df["duration"] = df["time"].astype(float)
df["event"] = df["DEATH_EVENT"].astype(int)
df["ef_low"] = (df["ejection_fraction"] < 35).astype(int)
df["creatinine_high"] = (df["serum_creatinine"] > 1.5).astype(int)
df["sodium_low"] = (df["serum_sodium"] < 135).astype(int)
df["log_cpk"] = np.log1p(df["creatinine_phosphokinase"])
df.head()

In [ ]:
cox_vars=['age','ejection_fraction','serum_creatinine','serum_sodium','anaemia','diabetes','high_blood_pressure','sex','smoking','log_cpk']
cox_df=df[['duration','event']+cox_vars].copy()
cph=CoxPHFitter(penalizer=0.05).fit(cox_df,'duration','event')
cph.summary[['coef','exp(coef)','p']]

In [ ]:
ph=proportional_hazard_test(cph, cox_df, time_transform='rank')
ph.summary

In [ ]:
schoenfeld=cph.compute_residuals(cox_df, kind='schoenfeld')
schoenfeld.head()

In [ ]:
cox_ext=cox_df.copy()
cox_ext['ef_x_log_time']=cox_ext['ejection_fraction']*np.log(cox_ext['duration'].clip(lower=1))
cph_ext=CoxPHFitter(penalizer=0.05).fit(cox_ext,'duration','event')
cph_ext.summary[['coef','exp(coef)','p']]

In [ ]:
delta=cph.compute_residuals(cox_df, kind='delta_beta')
delta.abs().sum(axis=1).sort_values(ascending=False).head(10)

Diagnostyka PH jest wykonywana po estymacji modelu Coxa. Sygnał naruszenia dla frakcji wyrzutowej uzasadnia sprawdzenie modelu z efektem zależnym od czasu.